# Week 6 Lab 4: Experiment Tracking and Champion Selection with MLflow

**Estimated duration:** about 120 minutes
**Scenario:** Cordwell Home and Hardware

## Scenario

Cordwell Home and Hardware is rolling out an AI assistant for customer questions. The team behind it runs two kinds of evaluations:

1. An **intent classifier** that routes each question (returns, installation help, pricing, and so on) to the right workflow.
2. **Prompt variants** for the answer generator, scored against reference answers with the metrics you learned earlier this week.

Until now every result lived in a spreadsheet nobody trusts. Your job is to put **MLflow** at the center of this workflow so every experiment records its parameters, metrics, tags, and artifacts, and so the team can pick a champion configuration with evidence instead of memory.

## Learning objectives

By the end of this lab you can:

1. Stand up a local **MLflow tracking server** with Docker and point a notebook at it.
2. Log **parameters, metrics, tags, artifacts, and a model** for a classifier experiment.
3. Build a **generation evaluation harness** that scores prompt variants with BLEU, ROUGE-L, and token F1, and logs every run.
4. Use the **MLflow UI** to filter, sort, and compare runs.
5. Apply an explicit **promotion policy** in code and tag a champion run through the MLflow client API.

## Timeline (about 120 minutes)

| Minutes | Part |
|---|---|
| 0 to 15 | Part 0: environment and MLflow server |
| 15 to 25 | Part 1: the Cordwell corpus |
| 25 to 45 | Part 2: generation metrics (TODO 1) |
| 45 to 70 | Part 3: tracked classifier experiments (TODO 2) |
| 70 to 95 | Part 4: tracked generation experiments (TODO 3) |
| 95 to 115 | Part 5: UI comparison and champion selection (TODO 4) |
| 115 to 120 | Wrap up and checkpoint |

Four TODO functions carry the lesson. Everything else (corpus generation, plotting, the offline answer simulator) is pre written so your time goes to the tracking concepts, not plumbing.


## Part 0: Environment and MLflow server (15 min)

### 0.1 Start the MLflow server with Docker

See instructions in **setup/LOCAL_MODE_SETUP.md**.

### 0.2 Backends for the generation part

The generation experiments default to a deterministic **offline** simulator, so the whole lab runs with no model server. If you want live answers, start LM Studio or Ollama with your pulled Gemma model and set the environment variables before launching Jupyter:

```bash
export BACKEND_MODE=lmstudio    # or ollama
export LOCAL_MODEL_NAME=gemma4  # confirm the exact tag your machine pulled
```

Both backends speak the OpenAI chat API, so one client covers them; only the port and model tag differ. Note that with a live backend the numeric soft checks switch to range checks, because live model output is not deterministic.


In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import random
import urllib.request
from collections import Counter
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

import sacrebleu
from rouge_score import rouge_scorer

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Where the MLflow tracking server lives. The Docker container publishes port 5001.
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5001")
EXPERIMENT_NAME = "week06_lab04_cordwell_tracking"

# Backend selection for the generation part of the lab.
# offline  = deterministic simulated answers, no model server needed (default)
# lmstudio = local LM Studio server on port 1234
# ollama   = local Ollama server on port 11434
BACKEND_MODE = os.getenv("BACKEND_MODE", "offline")

# Both LM Studio and Ollama speak the OpenAI chat API, so one client works for both.
# Confirm the exact model tag your machine has pulled before switching to a live backend.
MODEL_NAME = os.getenv("LOCAL_MODEL_NAME", "gemma4")

BACKEND_BASE_URLS = {
    "lmstudio": "http://127.0.0.1:1234/v1",
    "ollama": "http://127.0.0.1:11434/v1",
}

WORK_DIR = Path("lab_outputs")
WORK_DIR.mkdir(exist_ok=True)

print(f"Backend mode: {BACKEND_MODE}")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")

In [ ]:
MLFLOW_AVAILABLE = False
try:
    with urllib.request.urlopen(f"{MLFLOW_TRACKING_URI}/health", timeout=5) as resp:
        MLFLOW_AVAILABLE = resp.status == 200
except Exception as exc:
    print("Could not reach the MLflow tracking server.")
    print(f"  Tried: {MLFLOW_TRACKING_URI}/health")
    print(f"  Error: {type(exc).__name__}: {exc}")
    print()
    print("Fix: spin up a new MLflow container to host the platform, wait about")
    print("30 seconds for the server to finish starting, confirm the UI loads")
    print("at http://localhost:5001, then re-run this cell.")

if MLFLOW_AVAILABLE:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)
    print(f"MLflow server is up. Logging to experiment: {EXPERIMENT_NAME}")

In [ ]:
CHECK_RESULTS = {}


def run_check(name, fn):
    """Soft check runner: records PASS, FAIL, or TODO instead of crashing the notebook."""
    try:
        fn()
    except NotImplementedError:
        CHECK_RESULTS[name] = ("TODO", "not implemented yet")
        print(f"[TODO] {name}: not implemented yet")
    except AssertionError as exc:
        CHECK_RESULTS[name] = ("FAIL", str(exc))
        print(f"[FAIL] {name}: {exc}")
    except Exception as exc:
        CHECK_RESULTS[name] = ("FAIL", f"{type(exc).__name__}: {exc}")
        print(f"[FAIL] {name}: {type(exc).__name__}: {exc}")
    else:
        CHECK_RESULTS[name] = ("PASS", "")
        print(f"[PASS] {name}")


TODO_STEPS = set()


def attempt(label, fn):
    """Run a lab step that depends on a TODO. Degrades cleanly if the TODO is unfinished."""
    try:
        result = fn()
    except NotImplementedError:
        TODO_STEPS.add(label)
        print(f"[TODO] {label}: implement the function above, then re-run this cell.")
    except Exception as exc:
        print(f"[BLOCKED] {label}: {type(exc).__name__}: {exc}")
    else:
        TODO_STEPS.discard(label)
        return result


def require_mlflow():
    if not MLFLOW_AVAILABLE:
        raise AssertionError(
            "MLflow server is not reachable. Create a new docker container running MLflow in the lab, "
            "wait for http://localhost:5001 to load, then re-run the setup cells."
        )


def score_summary():
    total = len(CHECK_RESULTS)
    passed = sum(1 for status, _ in CHECK_RESULTS.values() if status == "PASS")
    print(f"Checks passed: {passed} of {total}")
    for name, (status, msg) in CHECK_RESULTS.items():
        line = f"  [{status}] {name}"
        if msg:
            line += f": {msg}"
        print(line)

## Part 1: The Cordwell corpus (10 min)

The next cell builds a seeded synthetic corpus of 500 customer questions. Each row has:

| Column | Meaning |
|---|---|
| `id` | Stable row id, also used to seed the offline simulator |
| `store_city`, `department` | Slicing fields for later error analysis |
| `question` | The customer's free text question |
| `intent` | The label the routing classifier should predict |
| `reference_answer` | What an ideal support agent would say |

About 30 percent of the questions are deliberately ambiguous between intents ("How much does it cost and can you install it?"), so the classifier cannot reach a perfect score and the confusion matrix has something to show.

Read the generator briefly, run it, and skim a few rows. You will not modify it.


In [ ]:
INTENT_LABELS = [
    "product_info",
    "installation_help",
    "returns_warranty",
    "pricing_promotions",
    "store_services",
    "delivery_pickup",
    "project_advice",
]

PRODUCTS = [
    "laminate flooring", "vinyl plank flooring", "hardwood flooring",
    "interior paint", "ceiling paint", "primer",
    "LED ceiling lights", "pendant lights", "bathroom vanity lights",
    "kitchen faucets", "bathroom faucets", "shower valves",
    "deck screws", "concrete mix", "drywall panels",
    "cordless drills", "impact drivers", "circular saws",
]

ROOMS = [
    "kitchen", "bathroom", "basement", "garage",
    "living room", "bedroom", "front porch", "back deck",
]

STORE_CITIES = ["Charlotte", "Raleigh", "Greensboro", "Asheville", "Wilmington", "Durham"]

DEPARTMENTS = [
    "flooring", "paint", "lighting", "plumbing", "hardware",
    "building materials", "tools", "outdoor",
]

AMBIGUOUS_PATTERNS = [
    ("Can I return {product} and get installation help for the replacement?",
     ["returns_warranty", "installation_help", "store_services"]),
    ("How much does {product} cost and can you install it?",
     ["pricing_promotions", "store_services", "installation_help"]),
    ("I need {product} for my {room}, do you have it and what is the price?",
     ["product_info", "pricing_promotions"]),
    ("Where can I get {product} and when can it be delivered?",
     ["product_info", "delivery_pickup"]),
    ("Can someone help me pick the right {product} for my {room} project?",
     ["product_info", "project_advice", "store_services"]),
    ("I want to buy {product}, any deals and can you deliver?",
     ["pricing_promotions", "delivery_pickup"]),
    ("What {product} do you recommend and how do I install it?",
     ["product_info", "installation_help", "project_advice"]),
    ("Do you sell {product} and offer installation services?",
     ["product_info", "store_services"]),
    ("Can I exchange {product} if it does not work for my {room}?",
     ["returns_warranty", "product_info"]),
    ("What is the best {product} and is it on sale?",
     ["product_info", "pricing_promotions", "project_advice"]),
]

QUESTION_TEMPLATES = {
    "product_info": [
        "Do you carry {product} suitable for a {room}?",
        "I need more details about {product} for my {room}. What are the options?",
        "Is {product} durable enough for a high traffic {room}?",
        "What kind of {product} do you have in stock?",
        "Tell me about the {product} options available.",
    ],
    "installation_help": [
        "What tools do I need to install {product} in my {room}?",
        "How hard is it to install {product} myself in the {room}?",
        "Do you have step by step instructions for installing {product}?",
        "Can I install {product} on my own or do I need professional help?",
        "What is involved in installing {product}?",
    ],
    "returns_warranty": [
        "Can I return {product} if I bought too much for my {room}?",
        "What is your return policy for opened {product}?",
        "I lost my receipt for {product}. Can I still return it?",
        "What is the policy on returning {product}?",
        "Can I exchange {product} for a different one?",
    ],
    "pricing_promotions": [
        "Do you offer bulk pricing on {product} for a large {room} project?",
        "Are there any current promotions on {product}?",
        "Can I stack coupons with a sale price on {product}?",
        "How much does {product} cost?",
        "Any discounts on {product} right now?",
    ],
    "store_services": [
        "Does Cordwell offer installation services for {product}?",
        "What time does the Cordwell near me close today?",
        "Can I get paint color matching for my {room}?",
        "What services do you offer for {product}?",
        "Can you help me with my {room} project in store?",
    ],
    "delivery_pickup": [
        "Can you deliver {product} to my home, or is it pickup only?",
        "What are the delivery windows for large orders of {product}?",
        "Do you offer same day pickup for online orders of {product}?",
        "How soon can I get {product} delivered?",
        "Is {product} available for curbside pickup?",
    ],
    "project_advice": [
        "I am remodeling my {room}. Should I use {product}, and what else should I consider?",
        "What is the best {product} option for a high traffic {room}?",
        "I am new to DIY. Can you help me plan a {room} refresh with {product}?",
        "What {product} works best for a {room} renovation?",
        "I am planning a {room} project. Any advice on using {product}?",
    ],
}

REFERENCE_TEMPLATES = {
    "product_info": (
        "Yes, Cordwell carries several {product} options for a {room}. "
        "You can filter by durability rating, finish, and customer reviews. "
        "I recommend checking the spec sheet for traffic rating and warranty details."
    ),
    "installation_help": (
        "For installing {product} in a {room}, you typically need basic carpentry tools, "
        "a level, measuring tape, and safety gear. Cordwell provides printed guides and "
        "online how to videos. You can also rent specialty tools from our tool rental center."
    ),
    "returns_warranty": (
        "Cordwell typically accepts returns within 90 days with proof of purchase. "
        "Some items like tinted paint or custom cut materials may be non returnable. "
        "Please bring your {product} and any order details to the service desk for review."
    ),
    "pricing_promotions": (
        "Cordwell often offers bulk pricing on items like {product}, especially for larger projects. "
        "Check our weekly ad and Pro benefits for current discounts. Coupons usually cannot be "
        "combined with existing promotional pricing, but your local store can confirm."
    ),
    "store_services": (
        "Most Cordwell locations offer services like professional installation, color matching, "
        "and project consultations. Store hours vary by location, so use the store finder to see "
        "today's hours and available services at your nearest store."
    ),
    "delivery_pickup": (
        "Cordwell offers home delivery and in store or curbside pickup for many items like {product}. "
        "Delivery windows depend on your ZIP code and order size. At checkout, you can choose delivery "
        "or pickup options and see the earliest available times."
    ),
    "project_advice": (
        "For a {room} project using {product}, consider traffic levels, moisture, and your skill level. "
        "Cordwell project planners can help you choose materials, calculate quantities, and create a "
        "step by step plan. You may also want to budget for underlayment, fasteners, and finishing materials."
    ),
}


def build_corpus(num_rows: int = 500, seed: int = SEED) -> pd.DataFrame:
    """Generate the synthetic Cordwell customer question corpus.

    Each row has a question, the intent label the routing classifier should predict,
    and the reference answer an ideal support agent would give. About 30 percent of
    questions are deliberately ambiguous between intents so the classifier cannot
    reach a perfect score.
    """
    rng = random.Random(seed)
    rows = []
    for i in range(num_rows):
        product = rng.choice(PRODUCTS)
        room = rng.choice(ROOMS)
        city = rng.choice(STORE_CITIES)
        dept = rng.choice(DEPARTMENTS)

        if rng.random() < 0.30:
            pattern, possible_intents = rng.choice(AMBIGUOUS_PATTERNS)
            intent = rng.choice(possible_intents)
            question = pattern.format(product=product, room=room)
        else:
            intent = rng.choice(INTENT_LABELS)
            question = rng.choice(QUESTION_TEMPLATES[intent]).format(product=product, room=room)

        reference = REFERENCE_TEMPLATES[intent].format(product=product, room=room)

        rows.append({
            "id": i,
            "store_city": city,
            "department": dept,
            "question": question,
            "intent": intent,
            "reference_answer": reference,
        })
    return pd.DataFrame(rows)


corpus_df = build_corpus()
corpus_path = WORK_DIR / "cordwell_eval_corpus.csv"
corpus_df.to_csv(corpus_path, index=False)
print(f"Corpus rows: {len(corpus_df)}")
print(f"Saved to: {corpus_path}")

In [ ]:
print("Intent distribution:")
print(corpus_df["intent"].value_counts())
print()
print("Three sample rows:")
for _, row in corpus_df.sample(3, random_state=SEED).iterrows():
    print(f"  [{row['intent']}] {row['question']}")

## Part 2: Generation metrics (20 min): TODO 1

Before any MLflow logging, you need one function that turns a batch of (reference, candidate) answer pairs into the three metrics this team tracks: **corpus BLEU**, **mean ROUGE-L F1**, and **mean token F1**.

The `token_f1` helper below is provided in full, including how `Counter` caps repeated words. Read it once; you will reuse it inside TODO 1.

### The two gotchas this TODO exists to teach

Both were covered on the Module 2 slides, and both fail **silently** if you get them wrong:

1. **sacrebleu** puts the **candidate first**: `sacrebleu.corpus_bleu(candidates, [references])`. Its score is on a **0 to 100 scale** by default, not 0 to 1.
2. **rouge-score** puts the **reference first**: `scorer.score(reference, candidate)`. This is the reverse of sacrebleu. Swapping the arguments swaps precision and recall.

### Worked target output

Here is exactly what a correct implementation produces for one toy pair, so you can code toward a visible target. The next cell runs this live so you can compare:

```text
sacrebleu corpus_bleu score (0 to 100 scale): 15.8542
  brevity penalty: 0.8825
rougeL precision: 0.7500
rougeL recall:    0.6667
rougeL f1:        0.7059
token_f1: 0.8235
```

For a batch, your `compute_generation_metrics` returns a dict like:

```python
{"bleu": 29.141083, "rouge_l_f1": 0.660508, "token_f1": 0.737163}
```

BLEU is corpus level (one number for the whole batch). ROUGE-L F1 and token F1 are computed per pair and then averaged.


In [ ]:
def token_f1(reference: str, candidate: str) -> float:
    """Token level F1: word overlap between candidate and reference.

    Counter (imported at the top of the notebook) counts how many times each
    token appears. A token counts as a true positive up to the number of times
    it appears in BOTH texts, so repeated words are not over counted.
    """
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()
    if not ref_tokens or not cand_tokens:
        return 0.0

    ref_counts = Counter(ref_tokens)
    cand_counts = Counter(cand_tokens)
    true_positives = sum(min(cand_counts[t], ref_counts[t]) for t in cand_counts)
    if true_positives == 0:
        return 0.0

    precision = true_positives / len(cand_tokens)
    recall = true_positives / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

In [ ]:
worked_reference = "we offer professional installation on flooring carpet and countertops"
worked_candidate = "we offer installation services on flooring and carpet"

# sacrebleu: the CANDIDATE goes first, references second, and the score is on a
# 0 to 100 scale by default. Getting the order backwards changes the number silently.
worked_bleu = sacrebleu.corpus_bleu([worked_candidate], [[worked_reference]])
print(f"sacrebleu corpus_bleu score (0 to 100 scale): {worked_bleu.score:.4f}")
print(f"  brevity penalty: {worked_bleu.bp:.4f}")

# rouge-score: score(target, prediction). The REFERENCE goes first here, which is
# the opposite of sacrebleu. Reversing the arguments swaps precision and recall.
worked_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
worked_rouge = worked_scorer.score(worked_reference, worked_candidate)["rougeL"]
print(f"rougeL precision: {worked_rouge.precision:.4f}")
print(f"rougeL recall:    {worked_rouge.recall:.4f}")
print(f"rougeL f1:        {worked_rouge.fmeasure:.4f}")

print(f"token_f1: {token_f1(worked_reference, worked_candidate):.4f}")

In [ ]:
def compute_generation_metrics(references: list, candidates: list) -> dict:
    """Compute corpus BLEU, mean ROUGE-L F1, and mean token F1 for paired texts.

    references[i] is the ideal answer for example i, candidates[i] is the model
    answer for the same example. Returns a dict with keys 'bleu' (0 to 100 scale),
    'rouge_l_f1' (0 to 1), and 'token_f1' (0 to 1).
    """
    assert len(references) == len(candidates), "references and candidates must pair up"
    assert len(references) > 0, "need at least one example"

    # sacrebleu corpus_bleu: candidates first, then a list of reference streams.
    bleu = sacrebleu.corpus_bleu(candidates, [references])

    # rouge-score: one scorer, then score(target, prediction) per pair.
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_vals = [
        scorer.score(ref, cand)["rougeL"].fmeasure
        for ref, cand in zip(references, candidates)
    ]

    token_f1_vals = [token_f1(ref, cand) for ref, cand in zip(references, candidates)]

    return {
        "bleu": float(bleu.score),
        "rouge_l_f1": float(np.mean(rouge_vals)),
        "token_f1": float(np.mean(token_f1_vals)),
    }

### Check your work

Run the summary cell any time. Checks report `PASS`, `FAIL`, or `TODO` without crashing the notebook.


In [ ]:
# Expected values for the soft checks. Every number below was computed by
# executing this exact code against the pinned libraries, not asserted from memory.
LOCKED_BLEU = 29.141082940320192
LOCKED_ROUGE = 0.660507792860734
LOCKED_TOKENF1 = 0.737163092661814
LOCKED_CLF_A_ACC = 0.84
LOCKED_GEN_METRICS = {
    "baseline_v1": {
        "bleu": 47.59503433605532,
        "rouge_l_f1": 0.6870281625623073,
        "token_f1": 0.6916686431685418,
        "avg_answer_tokens": 22.633333333333333,
    },
    "detailed_v2": {
        "bleu": 73.7567606720434,
        "rouge_l_f1": 0.8533266646124974,
        "token_f1": 0.8524887450329447,
        "avg_answer_tokens": 50.96666666666667,
    },
    "concise_v3": {
        "bleu": 28.771977064844688,
        "rouge_l_f1": 0.5198591283333011,
        "token_f1": 0.5281229493390575,
        "avg_answer_tokens": 20.3,
    },
}
LOCKED_CHAMPION_VARIANT = "detailed_v2"

def check_todo1():
    refs = [
        "we offer professional installation on flooring carpet and countertops",
        "most new unopened items can be returned within 90 days with proof of purchase",
        "store hours vary by location so use the store finder for today's hours",
    ]
    cands = [
        "we offer installation services on flooring and carpet",
        "unopened items can usually be returned within 90 days with a receipt",
        "hours vary by store so please check the store finder",
    ]
    m = compute_generation_metrics(refs, cands)
    assert set(m.keys()) == {"bleu", "rouge_l_f1", "token_f1"}, (
        f"expected keys bleu, rouge_l_f1, token_f1 but got {sorted(m.keys())}"
    )
    assert abs(m["bleu"] - LOCKED_BLEU) < 1e-6, (
        f"bleu={m['bleu']:.6f}, expected {LOCKED_BLEU:.6f}. "
        "Check the sacrebleu argument order (candidates first) and the 0 to 100 scale."
    )
    assert abs(m["rouge_l_f1"] - LOCKED_ROUGE) < 1e-6, (
        f"rouge_l_f1={m['rouge_l_f1']:.6f}, expected {LOCKED_ROUGE:.6f}. "
        "Check the rouge-score argument order: score(reference, candidate)."
    )
    assert abs(m["token_f1"] - LOCKED_TOKENF1) < 1e-6, (
        f"token_f1={m['token_f1']:.6f}, expected {LOCKED_TOKENF1:.6f}. "
        "Use the provided token_f1 helper and average it over the pairs."
    )


run_check("todo1_generation_metrics", check_todo1)

## Part 3: Tracked classifier experiments (25 min): TODO 2

Now the core of the lab. The classifier itself is standard scikit-learn from Labs 1.1 and 1.2, so `train_classifier`, the confusion matrix plotter, and the predictions CSV writer are all provided below. **Your work is the MLflow wrapper around them.**

### What goes where

| MLflow concept | In this experiment | Why there |
|---|---|---|
| **Parameters** | `max_features`, `clf_C`, ngram range, seed, corpus version | Inputs you chose before the run. Never change during a run. |
| **Metrics** | `val_accuracy`, `val_f1_weighted` | Numeric outcomes. Sortable and comparable in the UI. |
| **Tags** | `task`, `domain`, `dataset`, `owner` | Organizational labels for filtering, not inputs or outcomes. |
| **Artifacts** | confusion matrix PNG, predictions CSV | Files. Anything too rich for a single number. |
| **Model** | the fitted pipeline | Logged with `mlflow.sklearn.log_model(pipeline, name="intent_classifier")` |

One currency note baked into that last row: in MLflow 3 the model parameter is `name`, and `artifact_path` is deprecated. Logged models are also first class objects now. They appear under the experiment's **Models** view with an id starting with `m-`, not inside the run's artifact tree, so do not be surprised when the artifact browser shows only `plots` and `results`.

### Worked target output

A correct `run_classifier_experiment` prints one line and returns the run id string:

```text
Logged run 3f2a91bc (clf_bigram_C1.0): accuracy=0.840, f1=0.832
```

and afterwards the run in the UI shows 10 parameters, 2 metrics, 4 tags, artifact folders `plots` and `results`, and one logged model named `intent_classifier`. The check below verifies all of that through the client API.


In [ ]:
X = corpus_df["question"]
y = corpus_df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
print(f"Train rows: {len(X_train)}, test rows: {len(X_test)}")


def train_classifier(config: dict):
    """Train a TF-IDF plus logistic regression intent classifier.

    Returns (fitted_pipeline, metrics_dict, y_pred). This part is standard
    scikit-learn from earlier labs, so it is provided. Your work in this part
    is the MLflow logging around it.
    """
    pipeline = Pipeline(steps=[
        ("tfidf", TfidfVectorizer(
            max_features=config["max_features"],
            ngram_range=config["ngram_range"],
        )),
        ("clf", LogisticRegression(
            max_iter=config["max_iter"],
            C=config["C"],
        )),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    metrics = {
        "val_accuracy": float(accuracy_score(y_test, y_pred)),
        "val_f1_weighted": float(f1_score(y_test, y_pred, average="weighted")),
    }
    return pipeline, metrics, y_pred


def save_confusion_matrix_png(y_true, y_pred, run_label: str) -> str:
    """Render and save a confusion matrix heatmap. Returns the file path."""
    labels = sorted(INTENT_LABELS)
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color)
    ax.set_xlabel("Predicted intent")
    ax.set_ylabel("True intent")
    ax.set_title(f"Cordwell intent classifier: {run_label}")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()

    path = WORK_DIR / f"confusion_matrix_{run_label}.png"
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return str(path)


def save_predictions_csv(y_true, y_pred, questions, run_label: str) -> str:
    """Save per example predictions for later error analysis. Returns the file path."""
    frame = pd.DataFrame({
        "question": list(questions),
        "true_intent": list(y_true),
        "predicted_intent": list(y_pred),
    })
    frame["correct"] = frame["true_intent"] == frame["predicted_intent"]
    path = WORK_DIR / f"predictions_{run_label}.csv"
    frame.to_csv(path, index=False)
    return str(path)

In [ ]:
def run_classifier_experiment(config: dict) -> str:
    """Train one classifier configuration and log the full run to MLflow.

    Logs parameters, metrics, tags, two artifacts (confusion matrix plot and
    predictions CSV), and the fitted model. Returns the MLflow run id.
    """
    require_mlflow()

    with mlflow.start_run(run_name=config["run_name"]) as run:
        pipeline, metrics, y_pred = train_classifier(config)

        mlflow.log_params({
            "model_type": "tfidf_logreg",
            "max_features": config["max_features"],
            "ngram_min": config["ngram_range"][0],
            "ngram_max": config["ngram_range"][1],
            "clf_C": config["C"],
            "clf_max_iter": config["max_iter"],
            "train_size": len(X_train),
            "test_size": len(X_test),
            "random_seed": SEED,
            "corpus_version": "cordwell_synth_v1",
        })

        mlflow.set_tags({
            "task": "intent_classification",
            "domain": "home_improvement_retail",
            "dataset": "cordwell_synth_v1",
            "owner": "student",
        })

        mlflow.log_metrics(metrics)

        cm_path = save_confusion_matrix_png(y_test, y_pred, config["run_name"])
        mlflow.log_artifact(cm_path, artifact_path="plots")

        preds_path = save_predictions_csv(y_test, y_pred, X_test, config["run_name"])
        mlflow.log_artifact(preds_path, artifact_path="results")

        mlflow.sklearn.log_model(pipeline, name="intent_classifier")

        print(f"Logged run {run.info.run_id[:8]} ({config['run_name']}): "
              f"accuracy={metrics['val_accuracy']:.3f}, "
              f"f1={metrics['val_f1_weighted']:.3f}")
        return run.info.run_id

### Run two configurations

Two configs are defined below so you have something to compare in the UI. Run the cell after implementing TODO 2. Rerunning it creates fresh runs; that is normal MLflow behavior, and the check always validates the most recent pair.


In [ ]:
CLASSIFIER_CONFIGS = [
    {"run_name": "clf_bigram_C1.0", "max_features": 5000, "ngram_range": (1, 2), "C": 1.0, "max_iter": 400},
    {"run_name": "clf_unigram_C0.5", "max_features": 3000, "ngram_range": (1, 1), "C": 0.5, "max_iter": 400},
]

classifier_run_ids = {}


def _run_all_classifier_configs():
    for config in CLASSIFIER_CONFIGS:
        classifier_run_ids[config["run_name"]] = run_classifier_experiment(config)


attempt("classifier experiments", _run_all_classifier_configs)

In [ ]:
def check_todo2():
    if "classifier experiments" in TODO_STEPS:
        raise NotImplementedError
    require_mlflow()
    assert classifier_run_ids, (
        "no classifier runs recorded. Implement run_classifier_experiment and "
        "re-run the cell above this check."
    )
    client = MlflowClient()
    for run_name, run_id in classifier_run_ids.items():
        run = client.get_run(run_id)
        params = run.data.params
        metrics = run.data.metrics
        for key in ["model_type", "max_features", "clf_C", "corpus_version"]:
            assert key in params, f"run {run_name} is missing parameter '{key}'"
        for key in ["val_accuracy", "val_f1_weighted"]:
            assert key in metrics, f"run {run_name} is missing metric '{key}'"
            assert 0.0 <= metrics[key] <= 1.0, f"metric '{key}' out of range in {run_name}"
        assert run.data.tags.get("task") == "intent_classification", (
            f"run {run_name} is missing the task tag"
        )
        artifact_paths = {a.path for a in client.list_artifacts(run_id)}
        assert "plots" in artifact_paths, f"run {run_name} has no plots artifact folder"
        assert "results" in artifact_paths, f"run {run_name} has no results artifact folder"
        models = client.search_logged_models(experiment_ids=[run.info.experiment_id])
        assert any(m.source_run_id == run_id for m in models), (
            f"run {run_name} has no logged model. Use mlflow.sklearn.log_model."
        )
    acc_a = client.get_run(classifier_run_ids["clf_bigram_C1.0"]).data.metrics["val_accuracy"]
    assert abs(acc_a - LOCKED_CLF_A_ACC) < 1e-6, (
        f"clf_bigram_C1.0 accuracy={acc_a:.6f}, expected {LOCKED_CLF_A_ACC:.6f}. "
        "Make sure you pass the config values through to train_classifier unchanged."
    )


run_check("todo2_classifier_tracking", check_todo2)

## Part 4: Tracked generation experiments (25 min): TODO 3

Same tracking discipline, different experiment type. Three **prompt variants** for the Cordwell answer assistant are defined below, and every variant gets scored on the **same 60 sampled questions** so the comparison is paired, not apples to oranges.

Provided plumbing, already written:

- `simulate_answer` produces deterministic offline answers whose style mimics each variant (baseline drops detail, detailed keeps most content, concise truncates hard).
- `generate_llm_answer` calls whichever live backend `BACKEND_MODE` selects, through one shared OpenAI compatible client. A dead server raises a clear connection error instead of silently falling back.
- `answer_for_row` routes between the two, and `sample_eval_rows` returns the shared evaluation sample.

The smoke test cell shows one question answered by all three variants so you can see the style differences before scoring them.

### Worked target output

A correct `run_generation_experiment("baseline_v1")` with the offline backend prints:

```text
Logged run 8c14d0aa (gen_baseline_v1): bleu=47.60, rouge_l_f1=0.687, token_f1=0.692, avg_answer_tokens=22.6
```

and returns `(run_id, results_df)` where `results_df` is the sampled frame plus `model_answer` and `answer_tokens` columns. The logged run carries parameters (`prompt_version`, `temperature`, `sample_size`, `backend`, `model_name`, `corpus_version`), the tag `task=answer_generation`, all four metrics, and the per example CSV under `results`.


In [ ]:
PROMPT_VARIANTS = {
    "baseline_v1": {
        "description": "Short generic assistant. Tends to answer briefly and drop detail.",
        "system_prompt": (
            "You are a helpful but concise retail associate at Cordwell Home and Hardware. "
            "Answer customer questions in 2 to 3 sentences, focusing on the most important "
            "details only."
        ),
        "temperature": 0.3,
    },
    "detailed_v2": {
        "description": "Detailed assistant with emphasis on installation and policy clarity.",
        "system_prompt": (
            "You are a knowledgeable retail associate at Cordwell Home and Hardware. "
            "Provide clear, structured answers about products, installation services, and "
            "policies. Give 3 to 5 sentences and mention when availability or pricing may "
            "vary by store."
        ),
        "temperature": 0.3,
    },
    "concise_v3": {
        "description": "Extra concise assistant that redirects customers to the store quickly.",
        "system_prompt": (
            "You are a very concise associate at Cordwell Home and Hardware. Answer in one "
            "or two short sentences and point the customer to an in store expert for details."
        ),
        "temperature": 0.3,
    },
}

In [ ]:
def simulate_answer(reference: str, variant_id: str, row_id: int) -> str:
    """Deterministic offline stand in for a live model.

    Perturbs the reference answer in a way that mimics each prompt variant's
    style: baseline_v1 drops some detail, detailed_v2 keeps most content and
    adds filler, concise_v3 truncates hard. Seeded per row and variant so the
    same inputs always produce the same answer.
    """
    rng = random.Random(f"{row_id}-{variant_id}")
    sentences = [s for s in reference.split(". ") if s]

    if variant_id == "baseline_v1":
        keep = max(1, round(len(sentences) * 0.67))
        result = ". ".join(sentences[:keep])
        result = result.replace("typically", "usually")
        result = result.replace(", especially for larger projects", "")
        if rng.random() < 0.5:
            result = result.replace("Cordwell", "our store", 1)
    elif variant_id == "detailed_v2":
        keep = max(2, round(len(sentences) * 0.9))
        result = ". ".join(sentences[:keep])
        result = result.replace("We do offer", "We offer")
        if not result.endswith("."):
            result += "."
        result += " For complex projects, an associate in store can walk you through the options."
    elif variant_id == "concise_v3":
        keep = max(1, round(len(sentences) * 0.4))
        result = ". ".join(sentences[:keep])
        if not result.endswith("."):
            result += "."
        result += " Please visit your local store for details."
        result = result.replace("Yes, Cordwell carries", "Cordwell carries")
    else:
        result = ". ".join(sentences)

    if not result.endswith("."):
        result += "."
    return result


def get_openai_client():
    """One OpenAI compatible client covers both live backends.

    LM Studio and Ollama both expose the OpenAI chat completions API, so the
    only difference between them is the base URL and the model tag.
    """
    from openai import OpenAI

    if BACKEND_MODE not in BACKEND_BASE_URLS:
        raise ValueError(
            f"BACKEND_MODE is '{BACKEND_MODE}'. Live generation needs 'lmstudio' or 'ollama'."
        )
    return OpenAI(base_url=BACKEND_BASE_URLS[BACKEND_MODE], api_key="not-needed")


def generate_llm_answer(question: str, system_prompt: str, temperature: float,
                        max_tokens: int = 600) -> str:
    """Call the selected local backend. Raises a clear connection error if the
    server is not running rather than silently falling back."""
    client = get_openai_client()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content.strip()


def answer_for_row(row: dict, variant_id: str) -> str:
    """Route one question to the offline simulator or the live backend."""
    if BACKEND_MODE == "offline":
        return simulate_answer(row["reference_answer"], variant_id, row["id"])
    cfg = PROMPT_VARIANTS[variant_id]
    return generate_llm_answer(row["question"], cfg["system_prompt"], cfg["temperature"])


def sample_eval_rows(sample_size: int) -> pd.DataFrame:
    """Deterministic evaluation sample. Every variant is scored on the SAME rows,
    which makes the comparison between variants a paired comparison."""
    n = min(sample_size, len(corpus_df))
    return corpus_df.sample(n=n, random_state=SEED).reset_index(drop=True)

In [ ]:
_smoke_row = corpus_df.iloc[0].to_dict()
print("Question:")
print(f"  {_smoke_row['question']}")
print()
for _vid in PROMPT_VARIANTS:
    try:
        _ans = answer_for_row(_smoke_row, _vid)
        print(f"[{_vid}]")
        print(f"  {_ans}")
    except Exception as exc:
        print(f"[{_vid}] backend error: {type(exc).__name__}: {exc}")
    print()

In [ ]:
def run_generation_experiment(variant_id: str, sample_size: int = 60):
    """Evaluate one prompt variant and log the run to MLflow.

    Steps: sample the shared evaluation rows, produce an answer per row, score
    the answers with compute_generation_metrics, then log parameters, tags,
    metrics, and a per example CSV artifact. Returns (run_id, results_df).
    """
    require_mlflow()
    assert variant_id in PROMPT_VARIANTS, f"unknown variant: {variant_id}"
    cfg = PROMPT_VARIANTS[variant_id]

    sampled = sample_eval_rows(sample_size)

    with mlflow.start_run(run_name=f"gen_{variant_id}") as run:
        answers = [
            answer_for_row(row, variant_id)
            for row in sampled.to_dict("records")
        ]

        results_df = sampled.copy()
        results_df["model_answer"] = answers
        results_df["answer_tokens"] = [len(a.split()) for a in answers]

        metrics = compute_generation_metrics(
            results_df["reference_answer"].tolist(), answers
        )
        metrics["avg_answer_tokens"] = float(results_df["answer_tokens"].mean())

        mlflow.log_params({
            "prompt_version": variant_id,
            "temperature": cfg["temperature"],
            "sample_size": len(results_df),
            "backend": BACKEND_MODE,
            "model_name": MODEL_NAME if BACKEND_MODE != "offline" else "offline_simulation",
            "corpus_version": "cordwell_synth_v1",
        })

        mlflow.set_tags({
            "task": "answer_generation",
            "domain": "home_improvement_retail",
            "dataset": "cordwell_synth_v1",
            "owner": "student",
        })

        mlflow.log_metrics(metrics)

        csv_path = WORK_DIR / f"gen_results_{variant_id}.csv"
        results_df.to_csv(csv_path, index=False)
        mlflow.log_artifact(str(csv_path), artifact_path="results")

        print(f"Logged run {run.info.run_id[:8]} (gen_{variant_id}): "
              f"bleu={metrics['bleu']:.2f}, rouge_l_f1={metrics['rouge_l_f1']:.3f}, "
              f"token_f1={metrics['token_f1']:.3f}, "
              f"avg_answer_tokens={metrics['avg_answer_tokens']:.1f}")
        return run.info.run_id, results_df

In [ ]:
generation_run_ids = {}


def _run_all_generation_variants():
    for variant_id in PROMPT_VARIANTS:
        run_id, _ = run_generation_experiment(variant_id, sample_size=60)
        generation_run_ids[variant_id] = run_id


attempt("generation experiments", _run_all_generation_variants)

In [ ]:
def check_todo3():
    if "generation experiments" in TODO_STEPS:
        raise NotImplementedError
    require_mlflow()
    assert generation_run_ids, (
        "no generation runs recorded. Implement run_generation_experiment and "
        "re-run the cell above this check."
    )
    missing = set(PROMPT_VARIANTS) - set(generation_run_ids)
    assert not missing, (
        f"variants without a recorded run: {sorted(missing)}. Re-run the "
        "generation experiments cell after adding a variant."
    )
    client = MlflowClient()
    for variant_id, run_id in generation_run_ids.items():
        run = client.get_run(run_id)
        params = run.data.params
        metrics = run.data.metrics
        assert params.get("prompt_version") == variant_id, (
            f"run for {variant_id} logs prompt_version={params.get('prompt_version')}"
        )
        assert params.get("backend") == BACKEND_MODE, (
            f"run for {variant_id} should log the backend parameter"
        )
        for key in ["bleu", "rouge_l_f1", "token_f1", "avg_answer_tokens"]:
            assert key in metrics, f"run for {variant_id} is missing metric '{key}'"
        assert run.data.tags.get("task") == "answer_generation", (
            f"run for {variant_id} is missing the task tag 'answer_generation'"
        )
        artifact_paths = {a.path for a in client.list_artifacts(run_id, "results")}
        assert any(p.endswith(".csv") for p in artifact_paths), (
            f"run for {variant_id} has no per example CSV under results"
        )
        if BACKEND_MODE == "offline":
            expected = LOCKED_GEN_METRICS.get(variant_id)
            if expected is None:
                continue  # extra variants added in stretch goals have no locked values
            for key, val in expected.items():
                assert abs(metrics[key] - val) < 1e-4, (
                    f"{variant_id} metric {key}={metrics[key]:.4f}, expected {val:.4f}. "
                    "Use sample_eval_rows(60) so every variant scores the same rows."
                )
        else:
            assert 0.0 < metrics["rouge_l_f1"] <= 1.0, (
                f"{variant_id} rouge_l_f1 out of range for a live backend"
            )


run_check("todo3_generation_tracking", check_todo3)

## Part 5: Compare in the UI, then select a champion (20 min)

### 5.1 Guided UI exploration

Open http://localhost:5001 and click into the `week06_lab04_cordwell_tracking` experiment. Work through these steps and note what you see:

1. **Filter.** In the search box above the runs table, enter:

   ```text
   tags.task = 'answer_generation'
   ```

   The classifier runs disappear. This is why the tag discipline in TODOs 2 and 3 matters: without tags, every query is a scroll.

2. **Sort.** Sort the runs by `rouge_l_f1` descending using its column header (add the column through the column picker if it is not visible). Which variant leads? Does the BLEU ordering agree?

3. **Compare.** Select the three generation runs with their checkboxes and press **Compare**. Look at where the parameter rows differ and how the metric values line up. Note the trade off between `rouge_l_f1` and `avg_answer_tokens`: the highest scoring variant is also the wordiest.

4. **Artifacts.** Open the top run, find the per example CSV under `results`, and skim a few rows. Aggregate metrics say which run wins; per example artifacts say **why**.

5. **Metric filters.** Try a threshold query:

   ```text
   tags.task = 'answer_generation' and metrics.rouge_l_f1 > 0.6
   ```

### 5.2 Champion selection: TODO 4

Clicking around is fine for exploration, but promotion decisions should be **code**: explicit thresholds, applied the same way every time, with the decision written back into the tracking server. That is TODO 4.

The `PromotionPolicy` dataclass below holds the thresholds and a `policy.evaluate(metrics)` method that returns `(passes, reasons)`. Your function queries the generation runs through `MlflowClient`, ranks them by `rouge_l_f1` descending, finds the first run that clears every threshold, and tags it with `champion=true`, `promoted_date`, and `promotion_policy`.

### Worked target output

```text
Champion: gen_detailed_v2 (9d31c7e2)
      run_name  rouge_l_f1    bleu  token_f1  avg_answer_tokens  passes_policy  is_champion
gen_detailed_v2      0.8533 73.7568    0.8525            50.9667           True         True
gen_baseline_v1      0.6870 47.5950    0.6917            22.6333          False        False
 gen_concise_v3      0.5199 28.7720    0.5281            20.3000          False        False
```

After running it, refresh the champion run in the UI and confirm the three new tags are there.


In [ ]:
@dataclass
class PromotionPolicy:
    """Thresholds a run must clear before it can be promoted to champion."""
    name: str
    rouge_l_f1_min: float
    bleu_min: float
    token_f1_min: float
    max_answer_tokens: Optional[float] = None
    min_answer_tokens: Optional[float] = None

    def evaluate(self, metrics: dict):
        """Return (passes, reasons). Reasons list every failed criterion."""
        reasons = []
        if metrics.get("rouge_l_f1", 0.0) < self.rouge_l_f1_min:
            reasons.append(
                f"rouge_l_f1 {metrics.get('rouge_l_f1', 0.0):.3f} below {self.rouge_l_f1_min}"
            )
        if metrics.get("bleu", 0.0) < self.bleu_min:
            reasons.append(f"bleu {metrics.get('bleu', 0.0):.2f} below {self.bleu_min}")
        if metrics.get("token_f1", 0.0) < self.token_f1_min:
            reasons.append(
                f"token_f1 {metrics.get('token_f1', 0.0):.3f} below {self.token_f1_min}"
            )
        avg_len = metrics.get("avg_answer_tokens", 0.0)
        if self.max_answer_tokens is not None and avg_len > self.max_answer_tokens:
            reasons.append(f"answers too long: {avg_len:.1f} above {self.max_answer_tokens}")
        if self.min_answer_tokens is not None and avg_len < self.min_answer_tokens:
            reasons.append(f"answers too short: {avg_len:.1f} below {self.min_answer_tokens}")
        return len(reasons) == 0, reasons


STANDARD_POLICY = PromotionPolicy(
    name="standard",
    rouge_l_f1_min=0.80,
    bleu_min=60.0,
    token_f1_min=0.80,
    max_answer_tokens=60.0,
    min_answer_tokens=20.0,
)

In [ ]:
def select_and_tag_champion(policy: PromotionPolicy):
    """Query the generation runs, apply the policy, and tag the best passing run.

    Ranks candidate runs by rouge_l_f1 descending. The first run that clears
    every policy threshold becomes the champion and receives the tags
    champion=true, promoted_date, and promotion_policy. Returns
    (champion_run_id or None, report_df).
    """
    require_mlflow()
    client = MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    assert experiment is not None, f"experiment {EXPERIMENT_NAME} not found"

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.task = 'answer_generation'",
        order_by=["metrics.rouge_l_f1 DESC"],
    )
    assert runs, "no generation runs found. Finish TODO 3 first."

    report_rows = []
    champion_run_id = None
    for run in runs:
        run_name = run.data.tags.get("mlflow.runName", run.info.run_id[:8])
        passes, reasons = policy.evaluate(run.data.metrics)
        if passes and champion_run_id is None:
            champion_run_id = run.info.run_id
            client.set_tag(champion_run_id, "champion", "true")
            client.set_tag(champion_run_id, "promoted_date", date.today().isoformat())
            client.set_tag(champion_run_id, "promotion_policy", policy.name)
        report_rows.append({
            "run_name": run_name,
            "run_id": run.info.run_id,
            "rouge_l_f1": run.data.metrics.get("rouge_l_f1"),
            "bleu": run.data.metrics.get("bleu"),
            "token_f1": run.data.metrics.get("token_f1"),
            "avg_answer_tokens": run.data.metrics.get("avg_answer_tokens"),
            "passes_policy": passes,
            "is_champion": run.info.run_id == champion_run_id,
            "reasons": "; ".join(reasons) if reasons else "all criteria met",
        })

    report_df = pd.DataFrame(report_rows)
    return champion_run_id, report_df

In [ ]:
champion_result = {}


def _select_champion():
    run_id, report = select_and_tag_champion(STANDARD_POLICY)
    champion_result["run_id"] = run_id
    champion_result["report"] = report
    if run_id is None:
        print("No run cleared the policy. Report below.")
    else:
        winner = report.loc[report["run_id"] == run_id, "run_name"].iloc[0]
        print(f"Champion: {winner} ({run_id[:8]})")
    print(report[["run_name", "rouge_l_f1", "bleu", "token_f1",
                  "avg_answer_tokens", "passes_policy", "is_champion"]].to_string(index=False))


attempt("champion selection", _select_champion)

In [ ]:
def check_todo4():
    if "champion selection" in TODO_STEPS:
        raise NotImplementedError
    require_mlflow()
    assert "report" in champion_result, (
        "champion selection has not produced a report. Implement "
        "select_and_tag_champion and re-run the cell above this check."
    )
    report = champion_result["report"]
    for col in ["run_name", "rouge_l_f1", "passes_policy", "is_champion"]:
        assert col in report.columns, f"report is missing the '{col}' column"
    run_id = champion_result["run_id"]
    client = MlflowClient()
    if BACKEND_MODE == "offline":
        assert run_id is not None, (
            "with the offline backend, one variant clears the standard policy. "
            "Check that you rank by rouge_l_f1 descending and apply policy.evaluate."
        )
        run = client.get_run(run_id)
        assert run.data.params.get("prompt_version") == LOCKED_CHAMPION_VARIANT, (
            f"champion is {run.data.params.get('prompt_version')}, expected "
            f"{LOCKED_CHAMPION_VARIANT} under the standard policy with offline answers."
        )
    if run_id is not None:
        run = client.get_run(run_id)
        assert run.data.tags.get("champion") == "true", (
            "the champion run is missing the tag champion=true"
        )
        assert "promoted_date" in run.data.tags, "missing promoted_date tag"
        assert "promotion_policy" in run.data.tags, "missing promotion_policy tag"
        passes, reasons = STANDARD_POLICY.evaluate(run.data.metrics)
        assert passes, f"tagged champion does not clear the policy: {reasons}"


run_check("todo4_champion_selection", check_todo4)

## Stretch goals (optional, for fast finishers)

Full solutions to all three are in the instructor solution notebook, released after the lab.

1. **A fourth prompt variant.** Add a `friendly_v4` entry to `PROMPT_VARIANTS` with a warmer tone, run it through your harness, and see where it lands in the ranking. Careful: the offline simulator's fallback branch returns the reference almost unchanged, so an unknown variant scores nearly perfect. Add a matching branch to `simulate_answer` first, or your new variant cheats the metrics. That trap is worth thirty seconds of thought: it is exactly how evaluation harnesses produce numbers that are too good to be true.

2. **A semantic similarity metric.** N gram metrics punish honest paraphrases. Add a TF-IDF cosine similarity between each answer and its reference, log it as an extra metric in a new run, and check how it correlates with ROUGE-L on a scatter plot. (Sentence embedding models are the production grade version of this idea; TF-IDF cosine keeps this lab offline.)

3. **A policy sweep.** Define lenient and strict variants of the promotion policy and produce a report showing which runs pass at each level, plus a bar chart of pass counts. This is the shape of a real promotion gate: thresholds are debated in review, not hard coded in someone's head.


## Stretch goal solutions (instructor only)

Full, executed solutions to the three stretch goals. Release after the lab.


### Stretch 1: fourth prompt variant

The key move is extending the simulator **before** running the variant. The wrapper keeps a reference to the original function and only intercepts the new variant id.

In [ ]:
# Stretch 1 solution: a fourth prompt variant with a matching simulator branch

PROMPT_VARIANTS["friendly_v4"] = {
    "description": "Warm, encouraging associate who keeps most detail and adds a friendly close.",
    "system_prompt": (
        "You are a friendly, encouraging associate at Cordwell Home and Hardware. "
        "Answer clearly in 3 to 4 sentences, keep the key facts, and close warmly."
    ),
    "temperature": 0.4,
}

_base_simulate_answer = simulate_answer


def simulate_answer(reference: str, variant_id: str, row_id: int) -> str:
    """Extend the simulator with a friendly_v4 branch.

    Without this branch the fallback path would return the reference nearly
    unchanged and the new variant would score almost perfectly. That is the
    harness trap the stretch description warned about.
    """
    if variant_id == "friendly_v4":
        sentences = [s for s in reference.split(". ") if s]
        keep = max(2, round(len(sentences) * 0.8))
        result = ". ".join(sentences[:keep])
        if not result.endswith("."):
            result += "."
        result += " We are happy to help, and our team is here if you have questions."
        result = result.replace("Yes, Cordwell carries", "Cordwell is glad to carry")
        return result
    return _base_simulate_answer(reference, variant_id, row_id)


friendly_run_id, friendly_results = run_generation_experiment("friendly_v4", sample_size=60)
generation_run_ids["friendly_v4"] = friendly_run_id

### Stretch 2: TF-IDF cosine similarity

A separate run name (`gen_detailed_v2_semantic`) keeps the enriched run distinguishable from the original in the UI. The scatter artifact answers the correlation question visually.

In [ ]:
# Stretch 2 solution: TF-IDF cosine similarity as a semantic style metric

from sklearn.metrics.pairwise import cosine_similarity


def tfidf_cosine_similarities(references: list, candidates: list) -> list:
    """Cosine similarity between each candidate and its own reference,
    in a TF-IDF space fitted on all the texts at once."""
    vectorizer = TfidfVectorizer().fit(references + candidates)
    ref_matrix = vectorizer.transform(references)
    cand_matrix = vectorizer.transform(candidates)
    return [
        float(cosine_similarity(cand_matrix[i], ref_matrix[i])[0, 0])
        for i in range(len(references))
    ]


def run_generation_experiment_with_semantic(variant_id: str, sample_size: int = 60):
    """Rerun one variant with the extra semantic metric logged alongside the
    originals, plus a correlation scatter as an artifact."""
    require_mlflow()
    cfg = PROMPT_VARIANTS[variant_id]
    sampled = sample_eval_rows(sample_size)

    with mlflow.start_run(run_name=f"gen_{variant_id}_semantic") as run:
        answers = [answer_for_row(row, variant_id) for row in sampled.to_dict("records")]
        references = sampled["reference_answer"].tolist()

        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
        per_rouge = [scorer.score(r, a)["rougeL"].fmeasure for r, a in zip(references, answers)]
        per_cosine = tfidf_cosine_similarities(references, answers)

        metrics = compute_generation_metrics(references, answers)
        metrics["avg_answer_tokens"] = float(np.mean([len(a.split()) for a in answers]))
        metrics["tfidf_cosine"] = float(np.mean(per_cosine))

        mlflow.log_params({
            "prompt_version": variant_id,
            "temperature": cfg["temperature"],
            "sample_size": len(sampled),
            "backend": BACKEND_MODE,
            "model_name": MODEL_NAME if BACKEND_MODE != "offline" else "offline_simulation",
            "corpus_version": "cordwell_synth_v1",
            "extra_metric": "tfidf_cosine",
        })
        mlflow.set_tags({
            "task": "answer_generation",
            "domain": "home_improvement_retail",
            "dataset": "cordwell_synth_v1",
            "owner": "student",
            "has_semantic_metric": "true",
        })
        mlflow.log_metrics(metrics)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(per_cosine, per_rouge, alpha=0.6)
        corr = float(np.corrcoef(per_cosine, per_rouge)[0, 1])
        ax.set_xlabel("TF-IDF cosine similarity")
        ax.set_ylabel("ROUGE-L F1")
        ax.set_title(f"{variant_id}: cosine vs ROUGE-L (r={corr:.3f})")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        scatter_path = WORK_DIR / f"semantic_scatter_{variant_id}.png"
        fig.savefig(scatter_path, dpi=120)
        plt.close(fig)
        mlflow.log_artifact(str(scatter_path), artifact_path="plots")

        print(f"Logged run {run.info.run_id[:8]} (gen_{variant_id}_semantic): "
              f"rouge_l_f1={metrics['rouge_l_f1']:.3f}, "
              f"tfidf_cosine={metrics['tfidf_cosine']:.3f}, r={corr:.3f}")
        return run.info.run_id, corr


semantic_run_id, semantic_corr = run_generation_experiment_with_semantic("detailed_v2")

### Stretch 3: policy sweep

The sweep reports without tagging. Only the chosen policy (TODO 4) writes champion tags; a sweep that auto tagged at every level would produce conflicting champions.

In [ ]:
# Stretch 3 solution: a policy sweep across three strictness levels

SWEEP_POLICIES = [
    PromotionPolicy(name="lenient", rouge_l_f1_min=0.60, bleu_min=40.0,
                    token_f1_min=0.60),
    STANDARD_POLICY,
    PromotionPolicy(name="strict", rouge_l_f1_min=0.90, bleu_min=80.0,
                    token_f1_min=0.90, max_answer_tokens=45.0,
                    min_answer_tokens=25.0),
]


def policy_sweep_report(policies: list) -> pd.DataFrame:
    """Evaluate every generation run against every policy. Report only,
    no tagging: sweeps inform the debate about thresholds, the chosen
    policy makes the actual promotion."""
    require_mlflow()
    client = MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.task = 'answer_generation'",
        order_by=["metrics.rouge_l_f1 DESC"],
    )
    rows = []
    for policy in policies:
        for run in runs:
            passes, reasons = policy.evaluate(run.data.metrics)
            rows.append({
                "policy": policy.name,
                "run_name": run.data.tags.get("mlflow.runName", run.info.run_id[:8]),
                "rouge_l_f1": run.data.metrics.get("rouge_l_f1"),
                "passes": passes,
                "reasons": "; ".join(reasons) if reasons else "all criteria met",
            })
    return pd.DataFrame(rows)


sweep_df = policy_sweep_report(SWEEP_POLICIES)
print(sweep_df[["policy", "run_name", "rouge_l_f1", "passes"]].to_string(index=False))

pass_counts = sweep_df.groupby("policy", sort=False)["passes"].sum()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(pass_counts.index, pass_counts.values, color=["#2f9e44", "#f08c00", "#e03131"])
ax.set_ylabel("Runs passing")
ax.set_title("Generation runs passing each policy level")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
sweep_path = WORK_DIR / "policy_sweep.png"
fig.savefig(sweep_path, dpi=120)
plt.close(fig)
print(f"Saved chart to {sweep_path}")

## Wrap up and checkpoint

You have completed this lab when you can:

- Start the MLflow server with Docker and explain where its data lives.
- Say precisely what belongs in parameters vs metrics vs tags vs artifacts, and why.
- Point at a run in the UI and trace every number back to the code that logged it.
- Explain what your promotion policy requires and show the tagged champion run.

Two discussion prompts to close on with your table group:

1. The champion here won on `rouge_l_f1`, the metric we sorted by. What would change if the team sorted by BLEU? What does that say about picking your headline metric **before** running experiments?
2. If Cordwell replaced the offline simulator with a live model tomorrow, which parts of this notebook change and which stay exactly the same? (The answer to the second half is the point of the lab.)


In [ ]:
score_summary()